In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Pillar 3 – Tri-Partite Memory Kaggle Prototype

In [ ]:
# Step 1: basic imports and seed
import time, random
random.seed(42)

In [ ]:
# Step 2: define an event and generate one simulated day

from dataclasses import dataclass
from typing import List

@dataclass
class EpisodicEvent:
    raw_text: str
    day: int
    is_critical: bool
    importance: float
    who: str
    what: str
    where: str

def generate_day_events(day: int) -> List[EpisodicEvent]:
    """
    Create 100 events for one simulated day.
    ~10–15 are critical, the rest are noisy filler.
    """
    events = []
    
    # First, create 10–15 critical events
    num_critical = random.randint(10, 15)
    for i in range(num_critical):
        x, y, z = random.randint(-300, 300), 64, random.randint(-300, 300)
        raw = (
            f"CRITICAL Event {i} on day {day}: "
            f"Attacked by skeleton at {x}, {y}, {z}! Lost health and fought back."
        )
        events.append(
            EpisodicEvent(
                raw_text=raw,
                day=day,
                is_critical=True,
                importance=1.0,        # always high
                who="player",
                what="attacked_by_skeleton",
                where=f"{x},{y},{z}",
            )
        )
    
    # Then create the remaining noisy events
    while len(events) < 100:
        x, y, z = random.randint(-300, 300), 64, random.randint(-300, 300)
        raw = (
            f"NOISE Event on day {day}: "
            f"Walking around at {x}, {y}, {z}, mining dirt and looking at scenery."
        )
        importance = random.uniform(0.0, 1.0)
        events.append(
            EpisodicEvent(
                raw_text=raw,
                day=day,
                is_critical=False,
                importance=importance,
                who="player",
                what="walking",
                where=f"{x},{y},{z}",
            )
        )
    
    return events

# Quick smoke test: generate day 1 and print how many events and how many are critical
day1_events = generate_day_events(1)
print("Total events:", len(day1_events))
print("Critical events:", sum(1 for e in day1_events if e.is_critical))
print("Example event:", day1_events[0].raw_text[:120], "...")

In [ ]:
# Step 3: select which events to keep (important OR critical)

from typing import Dict

def select_events_for_consolidation(events):
    """
    Union-based selection:
    - Important events: importance > 0.7
    - Critical events: is_critical == True
    We return a list with every event that is important OR critical.
    """
    selected: Dict[int, EpisodicEvent] = {}

    # First, insert all important events
    for idx, e in enumerate(events):
        if e.importance > 0.7:
            selected[idx] = e

    # Then, insert all critical events (may overwrite same index, which is fine)
    for idx, e in enumerate(events):
        if e.is_critical:
            selected[idx] = e

    # The values() are the union of both sets
    return list(selected.values())

# Quick test on day 1
selected_day1 = select_events_for_consolidation(day1_events)
print("Selected events for consolidation:", len(selected_day1))
print("  of which critical:", sum(1 for e in selected_day1 if e.is_critical))

In [ ]:
# Step 4: simple compression and token counting (no LLM yet)

from dataclasses import dataclass

@dataclass
class SemanticFact:
    text: str
    who: str
    what: str
    where: str
    is_critical: bool

def simple_compress_event(event: EpisodicEvent) -> SemanticFact:
    """
    Placeholder 'compression' function.
    Later, you will replace the body of this function with a real 8B model call.
    For now, we just shrink the text manually.
    """
    # Very rough manual compression: who + what + where
    compressed = f"{event.who}:{event.what}@{event.where}"
    return SemanticFact(
        text=compressed,
        who=event.who,
        what=event.what,
        where=event.where,
        is_critical=event.is_critical,
    )

def count_tokens(text: str) -> int:
    """
    Extremely simple token approximation: split on spaces.
    Good enough for relative comparisons here.
    """
    return len(text.split())

def count_list_tokens(texts) -> int:
    return sum(count_tokens(t) for t in texts)

# Quick test on day 1
compressed_day1 = [simple_compress_event(e) for e in selected_day1]

raw_tokens_day1 = count_list_tokens([e.raw_text for e in day1_events])
compressed_tokens_day1 = count_list_tokens([f.text for f in compressed_day1])

print("Raw tokens (approx):", raw_tokens_day1)
print("Compressed tokens (approx):", compressed_tokens_day1)
compression_pct = 1 - (compressed_tokens_day1 / raw_tokens_day1)
print("Compression %:", round(compression_pct * 100, 2))

In [ ]:
# Step 6: run 10 simulated days and collect metrics

import pandas as pd

all_metrics = []

for day in range(1, 11):
    m = measure_day(day)
    all_metrics.append(m)
    print(f"Day {day}: "
          f"compression={m['compression_pct']:.2f}%, "
          f"sleep={m['sleep_latency_s']:.4f}s, "
          f"active_ctx={m['active_ctx_tokens']} tokens, "
          f"retention={m['retention_pct']:.1f}%")

df_metrics = pd.DataFrame(all_metrics)
df_metrics

In [ ]:
# Step 5: measure KPIs for a single day (no LLM yet)

def measure_day(day: int):
    # 1) Generate events
    events = generate_day_events(day)
    
    # 2) Count raw tokens (full diary)
    raw_tokens = count_list_tokens([e.raw_text for e in events])
    
    # 3) Select events for consolidation
    start_time = time.time()
    selected = select_events_for_consolidation(events)
    
    # 4) Compress selected events (placeholder compressor for now)
    compressed_facts = [simple_compress_event(e) for e in selected]
    sleep_latency_s = time.time() - start_time
    
    # 5) Count compressed tokens (semantic facts)
    compressed_tokens = count_list_tokens([f.text for f in compressed_facts])
    
    # 6) Active context = compressed semantic facts only (for now)
    active_ctx_tokens = compressed_tokens
    
    # 7) Critical fact retention
    critical_events = [e for e in events if e.is_critical]
    remembered = 0
    for ce in critical_events:
        # Check if any fact matches who+what+where
        for f in compressed_facts:
            if f.who == ce.who and f.what == ce.what and f.where == ce.where:
                remembered += 1
                break
    retention_pct = 100.0 * remembered / max(1, len(critical_events))
    
    # 8) Compression percentage
    compression_pct = 100.0 * (1 - compressed_tokens / max(1, raw_tokens))
    
    return {
        "day": day,
        "raw_tokens": raw_tokens,
        "compressed_tokens": compressed_tokens,
        "compression_pct": compression_pct,
        "sleep_latency_s": sleep_latency_s,
        "active_ctx_tokens": active_ctx_tokens,
        "critical_events": len(critical_events),
        "retention_pct": retention_pct,
    }

# Quick test for day 1
metrics_day1 = measure_day(1)
metrics_day1

In [ ]:
# Step 6: run 10 simulated days and collect metrics

import pandas as pd

all_metrics = []

for day in range(1, 11):
    m = measure_day(day)
    all_metrics.append(m)
    print(
        f"Day {day}: "
        f"compression={m['compression_pct']:.2f}%, "
        f"sleep={m['sleep_latency_s']:.4f}s, "
        f"active_ctx={m['active_ctx_tokens']} tokens, "
        f"retention={m['retention_pct']:.1f}%"
    )

df_metrics = pd.DataFrame(all_metrics)
df_metrics

In [ ]:
import os

root = "/kaggle/input/llama-3.1"
for dirpath, dirnames, filenames in os.walk(root):
    print("DIR:", dirpath)
    # print first few files in each dir
    for f in filenames[:5]:
        print("   FILE:", f)
    print("----")

In [ ]:
import os

print("Contents of /kaggle/input:")
print(os.listdir("/kaggle/input"))

In [ ]:
import os

root = "/kaggle/input/models"
print("Contents of /kaggle/input/models:")
print(os.listdir(root))

In [ ]:
import os

root = "/kaggle/input/models/metaresearch"
print("Contents of", root, ":", os.listdir(root))

In [ ]:
llama_root = "/kaggle/input/models/metaresearch/llama-3.1"
print("Contents of", llama_root, ":", os.listdir(llama_root))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/kaggle/input/models/metaresearch/llama-3.1/pytorch"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)
model.eval()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/kaggle/input/datasets/zhoumichael/meta-llama-3-8b-instruct/LLM-Research/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

model.eval()
print("Model loaded successfully")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/kaggle/input/datasets/zhoumichael/meta-llama-3-8b-instruct/LLM-Research/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

model.eval()
print("Model loaded successfully")

In [ ]:
from dataclasses import dataclass

@dataclass
class EpisodicEvent:
    raw_text: str
    day: int
    is_critical: bool
    importance: float
    who: str
    what: str
    where: str

@dataclass
class SemanticFact:
    text: str
    who: str
    what: str
    where: str
    is_critical: bool

def simple_compress_event(event: EpisodicEvent) -> SemanticFact:
    prompt = (
        "You are a compression engine for a Minecraft agent.\n"
        "Compress the event into ONE very short factual line.\n"
        "Keep who, what, and where.\n"
        "No explanation.\n\n"
        f"Event: {event.raw_text}\n"
        "Compressed:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Compressed:" in full_text:
        compressed = full_text.split("Compressed:", 1)[1].strip()
    else:
        compressed = full_text.strip()

    if not compressed:
        compressed = f"{event.who}:{event.what}@{event.where}"

    return SemanticFact(
        text=compressed,
        who=event.who,
        what=event.what,
        where=event.where,
        is_critical=event.is_critical,
    )

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import torch

model_path = "/kaggle/input/datasets/zhoumichael/meta-llama-3-8b-instruct/LLM-Research/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

model.eval()

# Tell Transformers what to use for padding and silence advisory logs
model.generation_config.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token_id = tokenizer.eos_token_id
transformers.logging.set_verbosity_error()

In [ ]:
# quick sanity test of simple_compress_event

test_event = EpisodicEvent(
    raw_text="CRITICAL Attacked by skeleton at 10, 64, -200! Lost 4 HP and fought back with a bow.",
    day=1,
    is_critical=True,
    importance=1.0,
    who="player",
    what="attacked_by_skeleton",
    where="10,64,-200",
)

fact = simple_compress_event(test_event)
print("Compressed fact:", fact.text)
print("who/what/where:", fact.who, fact.what, fact.where)

In [ ]:
# quick sanity test of simple_compress_event

test_event = EpisodicEvent(
    raw_text="CRITICAL Attacked by skeleton at 10, 64, -200! Lost 4 HP and fought back with a bow.",
    day=1,
    is_critical=True,
    importance=1.0,
    who="player",
    what="attacked_by_skeleton",
    where="10,64,-200",
)

fact = simple_compress_event(test_event)
print("Compressed fact:", fact.text)
print("who/what/where:", fact.who, fact.what, fact.where)

In [ ]:
all_metrics = []

for day in range(1, 11):
    m = measure_day(day)
    all_metrics.append(m)
    print(
        f"Day {day}: "
        f"compression={m['compression_pct']:.2f}%, "
        f"sleep={m['sleep_latency_s']:.4f}s, "
        f"active_ctx={m['active_ctx_tokens']} tokens, "
        f"retention={m['retention_pct']:.1f}%"
    )

df_metrics = pd.DataFrame(all_metrics)
df_metrics

In [ ]:
from typing import List, Dict

def generate_day_events(day: int, num_events: int = 20) -> List[EpisodicEvent]:
    events = []
    for i in range(num_events):
        is_critical = random.random() < 0.2
        importance = 1.0 if is_critical else random.uniform(0.2, 0.8)
        who = "player"
        what = random.choice([
            "mined_diamond",
            "mined_iron",
            "fought_zombie",
            "fought_skeleton",
            "explored_cave",
            "built_house",
            "crafted_tools",
            "traded_villager",
        ])
        where = f"{random.randint(-200, 200)},{random.randint(40, 80)},{random.randint(-200, 200)}"

        critical_prefix = "CRITICAL " if is_critical else ""
        raw_text = (
            f"{critical_prefix}Day {day}: {who} {what.replace('_', ' ')} "
            f"near {where}. Importance={importance:.2f}."
        )

        events.append(
            EpisodicEvent(
                raw_text=raw_text,
                day=day,
                is_critical=is_critical,
                importance=importance,
                who=who,
                what=what,
                where=where,
            )
        )
    return events

def compress_day_events(day: int, num_events: int = 20) -> Dict:
    events = generate_day_events(day, num_events=num_events)

    start_time = time.time()
    compressed_facts: List[SemanticFact] = []

    for ev in events:
        fact = simple_compress_event(ev)
        compressed_facts.append(fact)

    end_time = time.time()
    sleep_latency_s = end_time - start_time

    total_raw_tokens = 0
    total_compressed_tokens = 0

    for ev, fact in zip(events, compressed_facts):
        total_raw_tokens += len(ev.raw_text.split())
        total_compressed_tokens += len(fact.text.split())

    compression_pct = 100.0 * (1.0 - (total_compressed_tokens / max(total_raw_tokens, 1)))

    active_ctx_tokens = total_compressed_tokens
    retention_pct = 100.0

    return {
        "day": day,
        "num_events": len(events),
        "sleep_latency_s": sleep_latency_s,
        "compression_pct": compression_pct,
        "active_ctx_tokens": active_ctx_tokens,
        "retention_pct": retention_pct,
    }

def measure_day(day: int) -> Dict:
    return compress_day_events(day, num_events=20)

In [ ]:
from dataclasses import dataclass
from typing import List, Dict
import time
import random
import pandas as pd

@dataclass
class EpisodicEvent:
    raw_text: str
    day: int
    is_critical: bool
    importance: float
    who: str
    what: str
    where: str

@dataclass
class SemanticFact:
    text: str
    who: str
    what: str
    where: str
    is_critical: bool

def simple_compress_event(event: EpisodicEvent) -> SemanticFact:
    prompt = (
        "You are a compression engine for a Minecraft agent.\n"
        "Output ONE ultra-short fact (max 8 words).\n"
        "Keep who, what, and where.\n"
        "No explanation, no extra words.\n\n"
        f"Event: {event.raw_text}\n"
        "Compressed:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,      # shorter outputs than before
            do_sample=False
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Compressed:" in full_text:
        compressed = full_text.split("Compressed:", 1)[1].strip()
    else:
        compressed = full_text.strip()

    if not compressed:
        compressed = f"{event.who}:{event.what}@{event.where}"

    return SemanticFact(
        text=compressed,
        who=event.who,
        what=event.what,
        where=event.where,
        is_critical=event.is_critical,
    )

In [ ]:
# quick sanity test of simple_compress_event

test_event = EpisodicEvent(
    raw_text="CRITICAL Attacked by skeleton at 10, 64, -200! Lost 4 HP and fought back with a bow.",
    day=1,
    is_critical=True,
    importance=1.0,
    who="player",
    what="attacked_by_skeleton",
    where="10,64,-200",
)

fact = simple_compress_event(test_event)
print("Compressed fact:", fact.text)
print("who/what/where:", fact.who, fact.what, fact.where)